In [14]:
import importlib
import retriever
import engine

# Force Python to re-read the files from the disk
importlib.reload(retriever)
importlib.reload(engine)

# Now re-initialise Yves
yves = YvesEngine(config)
yves.sync_inventory()

Yves: Scanning the inventory... Don't mind the dust.
Sync complete. 0 files indexed.


In [6]:
import os
import sys
# Make sure we can see the src folder
sys.path.append(os.path.abspath(os.path.join('..')))

from src.config_loader import YvesConfig
config = YvesConfig()

inventory_path = config.INVENTORY_DIR
print(f"Checking folder: {inventory_path}")

# Check 1: Does the folder actually exist to Python?
if not os.path.exists(inventory_path):
    print("❌ ERROR: Folder does not exist!")
else:
    files_in_folder = os.listdir(inventory_path)
    print(f"📂 Raw files found in folder: {files_in_folder}")

    # Check 2: The Extension Filter
    # Most retrievers only look for specific types. Let's see what Yves is looking for.
    valid_extensions = ['.txt', '.py', '.md', '.pdf'] # Adjust based on your retriever.py
    
    indexed_count = 0
    for f in files_in_folder:
        ext = os.path.splitext(f)[1].lower()
        if ext in valid_extensions:
            print(f"✅ {f} is valid and will be indexed.")
            indexed_count += 1
        else:
            print(f"⚠️ {f} was SKIPPED (Unsupported extension: '{ext}')")

    print(f"\nFinal Result: Yves would index {indexed_count} files.")

Checking folder: C:\Users\Dusti\Downloads\Projects\YVES_CORE\data/inventory
📂 Raw files found in folder: ['test.txt']
✅ test.txt is valid and will be indexed.

Final Result: Yves would index 1 files.


In [7]:
def sync_yves():
    print("Yves is scanning the inventory...")
    # 1. Get all files
    files = manager.scan_inventory()
    
    for file_path in files:
        # 2. Chunk the file
        chunks = manager.read_and_chunk(file_path)
        
        # 3. Create metadata and IDs for each chunk
        metadata = [{"source": file_path} for _ in chunks]
        ids = [f"{os.path.basename(file_path)}_{i}" for i in range(len(chunks))]
        
        # 4. Store in the Vault
        vault.store_chunks(chunks, metadata, ids)
        
    print(f"Sync complete. Yves has memorised {len(files)} files.")

# Run this once you've put code in data/inventory/
# sync_yves()

In [15]:

from config_loader import YvesConfig
from engine import YvesEngine

config = YvesConfig()
yves = YvesEngine(config)

# Run once to index your files
yves.sync_inventory()

# Talk to her
print(yves.ask("Yves, look at my inventory and tell me if there's any messy code."))

Yves: Scanning the inventory... Don't mind the dust.
Sync complete. 0 files indexed.
The audit is on. Here's a review of your Python script and some notes on how to clean up the mess:

Your script uses ESLint for JavaScript and flake8 for Python, which is good practice. However, I notice you're using Radon for static code analysis, which is not necessary since it's primarily designed for Rust. For Python, you should use SonarQube or PyLint instead.

Let's focus on the code review:

1. **Code organization**: Your script mixes unrelated code snippets. It's time to separate concerns and put everything into its own file.
2. **Function length**: Some functions are too long. Break them down into smaller, manageable pieces. Remember, "one function per concern" is a good rule of thumb.
3. **Documentation**: Where is the docstring? Python requires it for complex modules or functions. Write it and make sure to include relevant information about what your code does.
4. **PEP 8 compliance**: Your 

In [2]:
import os

# Set the base directory to your project root (one level up from /notebooks)
BASE_DIR = os.path.abspath("..") 

# Define subdirectories
INVENTORY_DIR = os.path.join(BASE_DIR, "data", "inventory")
MEMORY_DIR = os.path.join(BASE_DIR, "data", "memory")
CONFIG_DIR = os.path.join(BASE_DIR, "config")
SRC_DIR = os.path.join(BASE_DIR, "src")

# Engineering check: Create folders if they don't exist
for folder in [INVENTORY_DIR, MEMORY_DIR, CONFIG_DIR, SRC_DIR]:
    os.makedirs(folder, exist_ok=True)

print(f"System Initialised.")
print(f"Yves Project Root: {BASE_DIR}")
print(f"Yves is watching: {INVENTORY_DIR}")

System Initialised.
Yves Project Root: C:\Users\Dusti\Downloads\Projects\YVES_CORE
Yves is watching: C:\Users\Dusti\Downloads\Projects\YVES_CORE\data\inventory


In [3]:
import os
#Config settings
class YvesConfig:
    def __init__(self):
        # Setting up the relative paths based on your structure
        self.BASE_DIR = os.path.abspath("..")
        self.INVENTORY_DIR = os.path.join(self.BASE_DIR, "data", "inventory")
        self.MEMORY_DIR = os.path.join(self.BASE_DIR, "data", "memory")
        self.CONFIG_PATH = os.path.join(self.BASE_DIR, "config", "yves_config.yaml")
        
        # Model Selection
        self.CODER_MODEL = "qwen2.5-coder:7b"
        self.CHAT_MODEL = "llama3.2:3b"
        
        # Ensure directories exist
        for path in [self.INVENTORY_DIR, self.MEMORY_DIR]:
            os.makedirs(path, exist_ok=True)

# Initialise it
config = YvesConfig()

In [4]:
#Inventory manager
class InventoryManager:
    def __init__(self, inventory_path):
        self.inventory_path = inventory_path
        self.supported_extensions = ('.py', '.js', '.cpp', '.h', '.html', '.css')

    def scan_inventory(self):
        """Finds all relevant code files in the inventory folder."""
        files_to_index = []
        for root, _, files in os.walk(self.inventory_path):
            for file in files:
                if file.endswith(self.supported_extensions):
                    files_to_index.append(os.path.join(root, file))
        return files_to_index

    def read_and_chunk(self, file_path, chunk_size=1000):
        """Reads a file and breaks it into smaller pieces for the AI."""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                # Simple chunking by character count (can be improved later)
                chunks = [content[i:i+chunk_size] for i in range(0, len(content), chunk_size)]
                return chunks
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            return []

# Test the Librarian
manager = InventoryManager(config.INVENTORY_DIR)
all_files = manager.scan_inventory()
print(f"Yves found {len(all_files)} files in the inventory.")

Yves found 0 files in the inventory.


In [5]:
import chromadb
from chromadb.utils import embedding_functions

class MemoryVault:
    def __init__(self, memory_path):
        # 1. Engineering check: We use PersistentClient so data is saved to disk
        self.client = chromadb.PersistentClient(path=memory_path)
        
        # 2. Define the 'Librarian' (Embedding Function)
        # This model turns text into numbers. all-MiniLM-L6-v2 is ultra-light.
        self.emb_fn = embedding_functions.DefaultEmbeddingFunction()
        
        # 3. Get or create the collection (like a table in SQL)
        self.collection = self.client.get_or_create_collection(
            name="yves_inventory", 
            embedding_function=self.emb_fn
        )

    def store_chunks(self, chunks, metadata_list, ids):
        """Stores code chunks into the vector database."""
        self.collection.add(
            documents=chunks,
            metadatas=metadata_list,
            ids=ids
        )

    def search(self, query, n_results=3):
        """Finds the most relevant pieces of code for a given question."""
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results
        )
        return results['documents'][0] if results['documents'] else []

# Initialise the Vault
vault = MemoryVault(config.MEMORY_DIR)